# Optimizers: SGD, Momentum, and Adam

This notebook accompanies the [Optimizers](https://codefrydev.in/Reinforcement/dl-foundations/optimizers/) page.
All implementations use NumPy only.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. SGD from scratch
Minimize f(w) = (w-3)^2 using SGD.

In [ ]:
def f(w): return (w - 3) ** 2
def grad_f(w): return 2 * (w - 3)

w = 0.0
lr = 0.1
history = [w]
for _ in range(30):
    w = w - lr * grad_f(w)
    history.append(w)

plt.plot(history)
plt.axhline(3, linestyle='--', color='gray', label='minimum')
plt.xlabel('Step'); plt.ylabel('w'); plt.title('SGD on f(w)=(w-3)^2')
plt.legend(); plt.show()
print(f'Final w: {history[-1]:.4f}, loss: {f(history[-1]):.6f}')


## 2. Momentum

In [ ]:
w_mom = 0.0
v = 0.0
beta = 0.9
lr = 0.1
mom_history = [w_mom]

for _ in range(30):
    g = grad_f(w_mom)
    v = beta * v - lr * g
    w_mom = w_mom + v
    mom_history.append(w_mom)

sgd_history = [0.0]
w = 0.0
for _ in range(30):
    w = w - lr * grad_f(w)
    sgd_history.append(w)

plt.plot(sgd_history, label='SGD')
plt.plot(mom_history, label='Momentum β=0.9')
plt.axhline(3, linestyle='--', color='gray', label='minimum')
plt.xlabel('Step'); plt.ylabel('w'); plt.title('SGD vs Momentum')
plt.legend(); plt.show()


## 3. Adam from scratch

In [ ]:
def adam_minimize(grad_fn, w_init=0.0, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, steps=30):
    w = w_init
    m, v = 0.0, 0.0
    history = [w]
    for t in range(1, steps + 1):
        g = grad_fn(w)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g ** 2
        m_hat = m / (1 - beta1 ** t)
        v_hat = v / (1 - beta2 ** t)
        w = w - lr * m_hat / (np.sqrt(v_hat) + eps)
        history.append(w)
    return w, history

w_adam, adam_history = adam_minimize(grad_f)

plt.plot(sgd_history, label='SGD')
plt.plot(mom_history, label='Momentum')
plt.plot(adam_history, label='Adam')
plt.axhline(3, linestyle='--', color='gray', label='minimum')
plt.xlabel('Step'); plt.ylabel('w'); plt.title('Optimizer Comparison')
plt.legend(); plt.show()
print('Adam final w:', round(w_adam, 4))


## 4. 2D function: elongated bowl
f(w1, w2) = w1^2 + 10*w2^2 — Adam handles different parameter scales better.

In [ ]:
def f2d(w): return w[0]**2 + 10*w[1]**2
def grad_f2d(w): return np.array([2*w[0], 20*w[1]])

# SGD
w_sgd = np.array([3.0, 3.0])
lr = 0.05
sgd_2d = [w_sgd.copy()]
for _ in range(50):
    w_sgd -= lr * grad_f2d(w_sgd)
    sgd_2d.append(w_sgd.copy())

# Adam
w_adam = np.array([3.0, 3.0])
m_a, v_a = np.zeros(2), np.zeros(2)
adam_2d = [w_adam.copy()]
for t in range(1, 51):
    g = grad_f2d(w_adam)
    m_a = 0.9 * m_a + 0.1 * g
    v_a = 0.999 * v_a + 0.001 * g**2
    m_hat = m_a / (1 - 0.9**t)
    v_hat = v_a / (1 - 0.999**t)
    w_adam -= 0.1 * m_hat / (np.sqrt(v_hat) + 1e-8)
    adam_2d.append(w_adam.copy())

sgd_loss = [f2d(w) for w in sgd_2d]
adam_loss = [f2d(w) for w in adam_2d]
plt.plot(sgd_loss, label='SGD'); plt.plot(adam_loss, label='Adam')
plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('SGD vs Adam on elongated bowl')
plt.legend(); plt.show()
